# Chapter 2: Working with text data

Code from Sebastian Raschka's *Build a Large Language Model (From Scratch)*. Each cell has the exercise, the book code it needs, and my code.

In [ ]:
# Setup
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
BOOK = ROOT / "resources/books/llm"
MODELS_DIR = str(ROOT / "work/llm/gpt2")


In [ ]:
# Exercise 2.1
# Try the BPE tokenizer from the tiktoken library on the unknown words "Akwirw ier" and
# print the individual token IDs. Then, call the decode function on each of the resulting
# integers in this list to reproduce the mapping shown in figure 2.11. Lastly, call the
# decode method on the token IDs to check whether it can reconstruct the original input,
# "Akwirw ier."

# Book code: ch02 section 2.5
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

# Alex's code starts here
# 1. First I need to install the library packages
# 2. Encode "Akwirw ier" using the BPE tokenizer from the tiktoken library
# 3. Print the individual token IDs
# 4. Decode each of the resulting ID to see the text it represents
# 5. Call the decode method on the token IDs to check if it can reconstruct the original input (string)

# Encode Akwirw ier using the BPE tokenizer from the tiktoken library
tokens = tokenizer.encode("Akwirw ier")

print(tokens)
for token in tokens:
    print(tokenizer.decode([token]))
print(tokenizer.decode(tokens))


In [ ]:
# Exercise 2.2
# Build intuition for the data loader by running it with other settings, such as
# max_length=2 with stride=2, and max_length=8 with stride=2.

# Book code: ch04 gpt.py, the chapter 2 data loader
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


with open(BOOK / "ch02/01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Alex's code starts here
